# Operaciones básicas de entrada/salida de archivos con Uproot

![uproot](img/uproot_logo.png)

# ¿Qué es Uproot?

Uproot es un paquete de Python que lee y escribe archivos ROOT, y está *únicamente* enfocado en la lectura y escritura (sin análisis, sin gráficos, etc.). Interactúa con NumPy, Awkward Array y Pandas para cálculos, boost-histogram/hist para manipulación y visualización de histogramas, Vector para funciones y transformaciones de vectores de Lorentz, Coffea para escalar el análisis, etc.

Uproot está implementado solo con Python y librerías de Python. No tiene una parte compilada ni requiere una versión específica de ROOT. (Esto significa que si *usas* ROOT para algo más que entrada/salida, tu elección de versión de ROOT no estará limitada por la entrada/salida).

![abstraction-layers](img/abstraction-layers.png)

Como consecuencia de ser una implementación independiente de la entrada/salida de ROOT, Uproot podría no ser capaz de leer/escribir ciertos tipos de datos. Cuáles tipos de datos no están implementados es un objetivo en constante movimiento, ya que siempre se están agregando nuevos. Una buena forma de leer datos es simplemente intentarlo y ver si Uproot genera algún error. Para la escritura, consulta las listas de tipos compatibles en la [documentación de Uproot](https://uproot.readthedocs.io/en/latest/basic.html#writing-objects-to-a-file) (cajas azules en el texto).

# Leer datos desde un archivo

## Abrir el archivo

Para abrir un archivo para lectura, pasa el nombre del archivo a [uproot.open](https://uproot.readthedocs.io/en/latest/uproot.reading.open.html). En los scripts, es una buena práctica usar la [instrucción with de Python](https://realpython.com/python-with-statement/) para cerrar el archivo cuando termines, pero si estás trabajando de forma interactiva, puedes usar una asignación directa.

In [ ]:
import skhep_testdata

nombre_del_archivo = skhep_testdata.data_path(
    "uproot-Event.root"
)  # descarga este archivo de prueba y obtiene una ruta local hacia él

import uproot

archivo = uproot.open(nombre_del_archivo)

Para acceder a un archivo remoto mediante HTTP o XRootD, utiliza una URL que comience con `"http://..."`, `"https://..."` o `"root://..."`. Si la interfaz de Python para XRootD no está instalada, el mensaje de error explicará cómo instalarla.

## Listar contenidos

Este objeto "`archivo`" en realidad representa un directorio, y los objetos nombrados en ese directorio son accesibles a través de una interfaz similar a un diccionario. Por lo tanto, `keys`, `values` e `items` devuelven los nombres de las claves y/o leen los datos. Si solo quieres listar los objetos sin leerlos, utiliza `keys`. (Esto es similar a `ls()` de ROOT, excepto que obtienes una lista de Python).

In [ ]:
archivo.keys()

A menudo también querrás conocer el tipo de cada objeto, por lo que los objetos [uproot.ReadOnlyDirectory](https://uproot.readthedocs.io/en/latest/uproot.reading.ReadOnlyDirectory.html) también tienen un método `classnames`, que devuelve un diccionario de nombres de objetos a nombres de clases (sin leerlos).

In [ ]:
archivo.classnames()

## Lectura de un histograma

Si estás familiarizado/a con ROOT, reconocerás `TH1F` como histogramas y `TTree` como un conjunto de datos. Para leer uno de los histogramas, coloca su nombre entre corchetes:

In [ ]:
h = archivo["hstat"]
h

Uproot no realiza ningún tipo de graficación ni manipulación de histogramas, por lo que los métodos más útiles de `h` comienzan con "to": `to_boost` (boost-histogram), `to_hist` (hist), `to_numpy` (la tupla de 2 elementos de NumPy que contiene el contenido y los bordes), `to_pyroot` (PyROOT), etc.

In [ ]:
h.to_hist().plot();

Los histogramas de Uproot también cumplen con el [protocolo de graficación UHI](https://uhi.readthedocs.io/en/latest/plotting.html), por lo que tienen métodos como `values` (contenidos de los bins), `variances` (errores al cuadrado) y `axes`.

In [ ]:
h.values()

In [ ]:
h.variances()

In [ ]:
list(h.axes[0])  # "x", "y", "z" o 0, 1, 2

## Lectura de un TTree

Un TTree representa un conjunto de datos potencialmente grande. Obtenerlo del [uproot.ReadOnlyDirectory](https://uproot.readthedocs.io/en/latest/uproot.reading.ReadOnlyDirectory.html) solo devuelve los nombres y tipos de sus TBranch. El método `show` es una forma conveniente de listar su contenido:

In [ ]:
t = archivo["T"]
t.show()

Ten en cuenta que puedes obtener la misma información de `keys` (un [uproot.TTree](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TTree.TTree.html) es similar a un diccionario), `typename` e `interpretation`.

In [ ]:
t.keys()

In [ ]:
t["event/fNtrack"], t["event/fNtrack"].typename, t["event/fNtrack"].interpretation

(Si un [uproot.TBranch](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TBranch.TBranch.html) no tiene `interpretation`, no se puede leer con Uproot).

La forma más directa de leer datos de un [uproot.TBranch](https://uproot.readthedocs.io/en/latest/uproot.behaviors.TBranch.TBranch.html) es llamando a su método `array`.

In [ ]:
t["event/fNtrack"].array()

Consideraremos otros métodos en la próxima lección.

## Leyendo un... ¿qué es eso?

Este archivo también contiene una instancia del tipo [TProcessID](https://root.cern.ch/doc/master/classTProcessID.html). Estos objetos no suelen ser útiles en el análisis de datos, pero Uproot logra leerlo de todos modos porque sigue ciertas convenciones (tiene "streamers de clase"). Se presenta como un objeto genérico con una propiedad `all_members` para sus miembros de datos (a través de todas las superclases).

In [ ]:
archivo["ProcessID0"]

In [ ]:
archivo["ProcessID0"].all_members

Aquí hay un ejemplo más útil de eso: una búsqueda de supernovas con el experimento IceCube tiene clases personalizadas para sus datos, que Uproot lee y representa como objetos con `all_members`.

In [ ]:
icecube = uproot.open(skhep_testdata.data_path("uproot-issue283.root"))
icecube.classnames()

In [ ]:
icecube["config/detector"].all_members

In [ ]:
icecube["config/detector"].all_members["ChannelIDMap"]

# Escribir datos en un archivo

La capacidad de Uproot para *escribir* datos es más limitada que su capacidad para *leer* datos, pero algunos casos útiles son posibles.

## Abrir archivos para escritura

Primero que nada, un archivo debe abrirse para escritura, ya sea creando un archivo completamente nuevo o actualizando uno existente.

In [ ]:
archivo_nuevo = uproot.recreate("archivo-completamente-nuevo.root")

```python
archivo_existente = uproot.update("archivo-existente.root")
```

(Uproot no puede escribir a través de una red; los archivos de salida deben ser locales).

## Escribir cadenas e histogramas

Estos objetos [uproot.WritableDirectory](https://uproot.readthedocs.io/en/latest/uproot.writing.writable.WritableDirectory.html) tienen una interfaz similar a un diccionario: puedes poner datos en ellos asignándolos a corchetes.

In [ ]:
archivo_nuevo["una_cadena"] = "Este objeto va a ser un TObjString."

archivo_nuevo["un_histograma"] = archivo["hstat"]

import numpy as np

archivo_nuevo["un_directorio/otro_histograma"] = np.histogram(
    np.random.normal(0, 1, 1000000)
)

En ROOT, el nombre de un objeto es una propiedad del objeto, pero en Uproot es una clave en el TDirectory que contiene el objeto, por eso el nombre está en el lado izquierdo de la asignación, entre corchetes. Solo se admiten los tipos de datos enumerados en la caja azul [de la documentación](https://uproot.readthedocs.io/en/latest/basic.html#writing-objects-to-a-file): principalmente solo histogramas.

## Escribir TTrees

Los TTrees son potencialmente grandes y pueden no caber en la memoria. Generalmente, necesitarás escribirlos en lotes.

```{warning}
Asignar datos a una clave del archivo, como en `archivo_nuevo["tree1"] = {"x": ..., "y": ...}`, *no* crea un TTree: las versiones recientes de Uproot escriben un RNTuple en ese caso (más sobre esto abajo). Usa `mktree` siempre que quieras específicamente un TTree.
```

Una forma de hacerlo es crear el TTree con [uproot.WritableDirectory.mktree](https://uproot.readthedocs.io/en/latest/uproot.writing.writable.WritableDirectory.html#mktree), pasándole el primer lote de datos, y extenderlo con `extend` usando los lotes siguientes:

In [ ]:
import numpy as np

archivo_nuevo.mktree(
    "tree1",
    {"x": np.random.randint(0, 10, 1000000), "y": np.random.normal(0, 1, 1000000)},
)
archivo_nuevo["tree1"].extend(
    {"x": np.random.randint(0, 10, 1000000), "y": np.random.normal(0, 1, 1000000)}
)
archivo_nuevo["tree1"].extend(
    {"x": np.random.randint(0, 10, 1000000), "y": np.random.normal(0, 1, 1000000)}
)

otra es pasar los tipos de datos en lugar de los datos mismos, lo que crea un TTree vacío, de modo que cada escritura sea una extensión.

In [ ]:
archivo_nuevo.mktree("tree2", {"x": np.int32, "y": np.float64})
archivo_nuevo["tree2"].extend(
    {"x": np.random.randint(0, 10, 1000000), "y": np.random.normal(0, 1, 1000000)}
)
archivo_nuevo["tree2"].extend(
    {"x": np.random.randint(0, 10, 1000000), "y": np.random.normal(0, 1, 1000000)}
)
archivo_nuevo["tree2"].extend(
    {"x": np.random.randint(0, 10, 1000000), "y": np.random.normal(0, 1, 1000000)}
)

En la próxima lección se dan consejos de rendimiento, pero en general vale la pena escribir pocos lotes grandes en lugar de muchos lotes pequeños.

Los únicos tipos de datos que se pueden pasar a `mktree` y `extend` están listados en la caja azul [de esta documentación](https://uproot.readthedocs.io/en/latest/basic.html#writing-ttrees-to-a-file). Esto incluye arrays irregulares (descritos en la lección de después de la próxima), pero no tipos más complejos.

# Leer y escribir RNTuples

Durante décadas, TTree ha sido el formato por omisión para almacenar conjuntos de datos grandes en archivos ROOT. Sin embargo, poco a poco ha quedado desactualizado y no está optimizado para los sistemas modernos. Ahí es donde entra el formato RNTuple. Es un formato de serialización moderno, diseñado pensando en los sistemas actuales, y está previsto que reemplace a TTree en los próximos años. La [versión 1.0.0.0](https://cds.cern.ch/record/2923186) ya salió y va a tener soporte "para siempre".

Los RNTuples son mucho más simples que los TTrees por diseño, y esta vez hay una especificación oficial, lo que hace mucho más fácil que paquetes de entrada/salida de terceros como Uproot puedan soportarlos. Uproot ya soporta la lectura de la especificación completa de RNTuple, lo que significa que puedes leer cualquier RNTuple que te encuentres. También soporta escribir una gran parte de la especificación, y la intención es soportar todo lo que tenga sentido para el análisis de datos.

Para facilitar la transición a los RNTuples, estamos diseñando la interfaz para que se parezca lo más posible a la de los TTrees. Veamos un ejemplo simple de lectura y escritura de RNTuples.

De nuevo, usaremos un archivo de ejemplo del paquete `scikit-hep-testdata`.

In [ ]:
nombre_del_archivo = skhep_testdata.data_path("test_stl_containers_rntuple_v1-0-0-0.root")

archivo = uproot.open(nombre_del_archivo)

Esta vez, si imprimimos los nombres de las clases, vemos que hay un RNTuple en lugar de un TTree.

In [ ]:
archivo.classnames()

Veamos las claves disponibles con `.keys`, pero restringiéndonos solo a las del nivel superior usando `recursive=False`.

In [ ]:
rntuple = archivo["ntuple"]
rntuple.keys(recursive=False)

Podemos mostrar la estructura del RNTuple más claramente usando `.show`, que funciona de manera similar a como lo hace con los TTrees, pero es más completo y muestra mejor la estructura, ya que los RNTuples son completamente legibles por Uproot.

In [ ]:
rntuple.show()

Leer datos en arrays funciona exactamente igual que para los TTrees, así que no tienes que preocuparte por distinguir si estás leyendo un TTree o un RNTuple.

In [ ]:
datos = rntuple.arrays()
datos

Escribir es incluso más simple que para los TTrees: basta con asignar los datos a una clave del archivo, de la misma manera pythónica que para las cadenas y los histogramas. A partir de Uproot 5.7, esto escribe un RNTuple (en versiones anteriores escribía un TTree), y el objeto resultante se puede extender con más lotes, igual que un TTree.

In [ ]:
datos = {"mis_datos_enteros": [1, 2, 3], "mis_datos_flotantes": [1.0, 2.0, 3.0]}
mas_datos = {"mis_datos_enteros": [4, 5, 6], "mis_datos_flotantes": [4.0, 5.0, 6.0]}

archivo_nuevo3 = uproot.recreate("archivo-nuevo-con-rntuple.root")

archivo_nuevo3["mi_rntuple"] = datos
archivo_nuevo3["mi_rntuple"].extend(mas_datos)

Si prefieres declarar los campos por adelantado y hacer que cada escritura sea una extensión, [uproot.WritableDirectory.mkrntuple](https://uproot.readthedocs.io/en/latest/uproot.writing.writable.WritableDirectory.html#mkrntuple) es la contraparte de `mktree` para RNTuples.

Para el resto del tutorial nos quedaremos principalmente con los TTrees, ya que este sigue siendo el formato de datos principal que te vas a encontrar en el futuro cercano.

`````{tip}
# Ejercicio 1 (10-15 minutos)

Hay un archivo en `skhep_testdata` llamado `ntpl001_staff_rntuple_v1-0-0-0.root` que contiene datos del personal del CERN de 1988. Como sugiere el nombre, es un RNTuple y no un TTree.

1.  Ábrelo con Uproot, mira lo que hay dentro, y luego encuentra el número de empleados franceses que tenían al menos 35 años y uno o dos hijos. Como punto extra, usa `with uproot.open(...) as f:` en lugar de `f = uproot.open(...)` para seguir las buenas prácticas.
2.  Con la selección de la parte anterior, haz un histograma del grado (`Grade`) de los empleados con `np.histogram(datos.to_numpy())`, y guárdalo en un archivo ROOT nuevo. (Convierte primero los datos con `.to_numpy()`: `np.histogram` de un Awkward Array devuelve Awkward Arrays en lugar de arrays de NumPy, y Uproot no reconocería ese par como un histograma).
3.  Vuelve a leer el archivo que acabas de crear. Abre el histograma, usa `to_hist()` para convertirlo en un histograma de `hist`, y luego grafícalo con `.plot()`.
4.  Si estás en un evento de formación, haz clic derecho sobre la imagen, luego en "Create New View for Cell Output", después clic derecho sobre la imagen en la nueva vista, luego "Copy Image", y pégala como respuesta en el canal de Slack.

````{note}
:class: dropdown
## Solución (¡no hagas trampa!)


```python
import skhep_testdata
import uproot
import numpy as np

with uproot.open(skhep_testdata.data_path("ntpl001_staff_rntuple_v1-0-0-0.root")) as archivo:
    staff = archivo["Staff"]
    edad = staff["Age"].array()
    nacion = staff["Nation"].array()
    hijos = staff["Children"].array()
    grado = staff["Grade"].array()

corte = ((nacion == "FR")
    & (edad >= 35)
    & (1 <= hijos)
    & (2 >= hijos))

n = len(edad[corte])

n = np.sum(corte)  # Una alternativa más simple es contar el número de valores True en corte

print(f"El número de empleados con los criterios seleccionados es {n}")

with uproot.recreate("mi_archivo.root") as archivo:
    archivo["mi_hist"] = np.histogram(grado[corte].to_numpy())

with uproot.open("mi_archivo.root") as archivo:
    h = archivo["mi_hist"].to_hist()

h.plot();
```
````
`````